# Agroforestry inventory from drone imagery

Counts and locates individual plants in an agroforestry system from a drone
orthophoto. You upload the image, choose a species, and get a map with every plant
marked plus a geographic file for your GIS.

**How to run:** *Runtime* menu -> *Run all*, then answer when it asks for the image.

> The code is hidden so it does not get in the way of reading. Click **Show code**
> on any step to audit it - everything is open, under GPL-3.0.
> Source: [github.com/COURAGEOUS-LAND/agroforestry-inventory](https://github.com/COURAGEOUS-LAND/agroforestry-inventory)

---

### Before trusting a number

**These models do not transfer from one place to another without loss.** That is
measured, not a disclaimer. One of our coffee models scored well on its own
validation set and, applied to held-out areas of the **same orthophoto** a few
hundred metres from where it was trained, found **2 of 198** plants. Its precision
stayed at 1.00 - the two it found were correct. It invented nothing:
**it went blind, and nothing in the output said so.**

So when you run it on your imagery: **look at the figure in step 6**. If the boxes
do not sit on the plants, the model does not work there. Count a small area by hand
and compare. We would rather say this on the first screen than let you find out
later.

## 1. Set up

In [ ]:
# @title Install dependencies &nbsp;·&nbsp; *takes 2 to 3 minutes, once per session*
%pip install -q rfdetr rasterio geopandas shapely scipy tqdm huggingface_hub folium

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} - detection will take about a minute")
else:
    print("No GPU (works the same, just slower: about 4 min per orthophoto)")
    print("To enable: Runtime -> Change runtime type -> GPU")

## 2. Choose the species

The species and the figures below come from the manifest published alongside the
weights - they are not copied into this notebook. What you read is what the model
actually is.

In [ ]:
# @title Species to detect
SPECIES = "coffee"  # @param ["coffee", "avocado", "pitaya", "banana"]

import json, urllib.request, hashlib
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO = "JefersonPMS/agroforestry-inventory"
manifest = json.loads(urllib.request.urlopen(
    f"https://huggingface.co/{REPO}/resolve/main/models.json").read())
card = next(m for m in manifest["models"] if m["species"] == SPECIES)

print(f"{card['title']}\n")
print(f"  trained at ........ {card['site']}")
print(f"  precision ......... {card['precision']}   of the plants reported, how many are real")
print(f"  recall ............ {card['recall']}   of the real plants, how many were found")
print(f"  position error .... {card['position_error_m']} m")
print(f"  median crown ...... {card['median_crown_m']} m")
print(f"  trained on ........ {card['training_labels']} hand-marked plants")
print(f"\n  WHERE THIS MODEL FAILS:\n  {card['where_it_fails']}")

model_path = hf_hub_download(repo_id=REPO, filename=card["file"])
# check the sha256: a download cut short leaves a truncated file that loads and
# infers wrongly, without complaining
assert hashlib.sha256(Path(model_path).read_bytes()).hexdigest() == card["sha256"], \
    "corrupted file - delete it and run again"
print(f"\n  model downloaded and verified ({Path(model_path).stat().st_size/1e6:.0f} MB)")

## 3. Your orthophoto

It must be a georeferenced GeoTIFF with at least three bands (RGB).
For a large file, Google Drive is much better than uploading through the browser.

In [ ]:
# @title Where the image comes from
SOURCE = "example"  # @param ["example", "my computer", "Google Drive"]
PATH_IN_DRIVE = "/content/drive/MyDrive/my_orthophoto.tif"  # @param {type:"string"}

if SOURCE == "example":
    import urllib.request
    ORTHOPHOTO = "example_coffee.tif"
    if not Path(ORTHOPHOTO).exists():
        urllib.request.urlretrieve(
            f"https://huggingface.co/{REPO}/resolve/main/example_coffee.tif", ORTHOPHOTO)
    print("A 34 x 34 m patch of an agroforestry system in Sao Paulo, Brazil, with coffee.")
    print("For your own image, change SOURCE above.")

elif SOURCE == "my computer":
    from google.colab import files
    ORTHOPHOTO = list(files.upload().keys())[0]

else:
    from google.colab import drive
    drive.mount("/content/drive")
    ORTHOPHOTO = PATH_IN_DRIVE

import rasterio
with rasterio.open(ORTHOPHOTO) as src:
    gsd = abs(src.transform.a)
    print(f"\n{ORTHOPHOTO}")
    print(f"  {src.width} x {src.height} px · {src.count} bands · {src.crs}")
    print(f"  resolution: {gsd*100:.1f} cm per pixel")
    if not 0.005 < gsd < 0.06:
        print("  WARNING: the models were trained at about 1.7 cm/px."
              " A very different resolution degrades the result.")

## 4. The detection engine

An orthophoto does not fit into a vision model in one piece, so it is walked in
overlapping tiles. Each box the model returns in pixel space becomes a geographic
coordinate through the raster transform. Because the tiles overlap, a plant near an
edge is detected twice - the de-duplication step merges those repeats and keeps the
most confident one.

In [ ]:
# @title Load the engine &nbsp;·&nbsp; *produces no output; only defines the functions*
import numpy as np, rasterio, time
from PIL import Image
from rasterio.windows import Window
from rasterio.transform import xy as raster_xy
from shapely.geometry import Polygon, Point
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

def to_rgb(tile):
    """Raster bands -> 8-bit RGB, stretched 2%-98% as in training."""
    planes = []
    for b in range(3):
        p = tile[b].astype(np.float32)
        v = p[~np.isnan(p)]
        if v.size == 0:
            planes.append(np.zeros(p.shape, np.uint8)); continue
        lo, hi = np.percentile(v, 2), np.percentile(v, 98)
        if hi - lo < 1e-3: hi = lo + 1
        planes.append(np.clip((p - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8))
    return np.stack(planes, -1)

def deduplicate(recs, dist_m, crs):
    """Same plant seen in neighbouring tiles: keep the most confident one."""
    if len(recs) <= 1: return recs
    import geopandas as gpd
    g = gpd.GeoDataFrame(geometry=[r['centroid'] for r in recs], crs=crs)
    try: metric = g.estimate_utm_crs()
    except Exception: metric = crs
    pts = np.array([[p.x, p.y] for p in g.to_crs(metric).geometry])
    tree, dead = cKDTree(pts), np.zeros(len(pts), bool)
    for i in np.argsort([-r['confidence'] for r in recs]):
        if dead[i]: continue
        for j in tree.query_ball_point(pts[i], r=dist_m):
            if j != i: dead[j] = True
    return [r for k, r in enumerate(recs) if not dead[k]]

def detect(orthophoto, model, label, confidence=0.25, dedup_m=0.5, tile_px=640, overlap=0.2):
    from rfdetr import RFDETRNano
    net = RFDETRNano(pretrain_weights=model)
    step = int(tile_px * (1 - overlap))
    recs, t0 = [], time.time()

    with rasterio.open(orthophoto) as src:
        assert src.count >= 3, f'the raster needs 3 RGB bands (it has {src.count})'
        n_c = max(1, (src.width - tile_px) // step + 1)
        n_r = max(1, (src.height - tile_px) // step + 1)
        print(f'{n_r*n_c} tiles of {tile_px} px')

        for r in tqdm(range(n_r), desc='progress'):
            for c in range(n_c):
                cx, cy = c * step, r * step
                data = src.read(window=Window(cx, cy, tile_px, tile_px))
                if data.shape[0] < 3: continue
                h, w = data.shape[1], data.shape[2]
                rgb = to_rgb(data[:3])
                if rgb.mean() < 12: continue          # outside the footprint
                buf = np.zeros((tile_px, tile_px, 3), np.uint8)
                buf[:h, :w] = rgb[:tile_px, :tile_px]

                det = net.predict(Image.fromarray(buf), threshold=confidence, shape=(384, 384))
                if det is None or len(det) == 0: continue
                for k in range(len(det.xyxy)):
                    x1, y1, x2, y2 = det.xyxy[k]
                    x1, x2 = np.clip([x1, x2], 0, w); y1, y2 = np.clip([y1, y2], 0, h)
                    if x2 <= x1 or y2 <= y1: continue
                    cols = np.array([x1, x2, x2, x1]) + cx
                    rows = np.array([y1, y1, y2, y2]) + cy
                    pts = [raster_xy(src.transform, ro, co, offset='ul') for ro, co in zip(rows, cols)]
                    poly = Polygon(pts)
                    if poly.is_empty or not poly.is_valid: continue
                    recs.append({'class': label, 'confidence': float(det.confidence[k]),
                                 'box': poly, 'centroid': Point(poly.centroid.x, poly.centroid.y)})
        crs = src.crs

    raw = len(recs)
    recs = deduplicate(recs, dedup_m, crs)
    print(f'{raw} raw detections -> {len(recs)} after removing repeats '
          f'({time.time()-t0:.0f}s)')
    return recs, crs

## 5. Detect

In [ ]:
# @title Run the detection
MINIMUM_CONFIDENCE = 0.25  # @param {type:"slider", min:0.05, max:0.9, step:0.05}

records, crs = detect(ORTHOPHOTO, model_path, card["species"],
                      confidence=MINIMUM_CONFIDENCE,
                      dedup_m=card["recommended_dedup_m"])

import geopandas as gpd
attrs = {"class": [r["class"] for r in records],
         "confidence": [r["confidence"] for r in records]}
boxes = gpd.GeoDataFrame(attrs, geometry=[r["box"] for r in records], crs=crs).to_crs(4326)
points = gpd.GeoDataFrame(attrs, geometry=[r["centroid"] for r in records], crs=crs).to_crs(4326)

print(f"\n{'='*46}")
print(f"  {len(boxes)} {card['title']} plants")
print(f"  confidence: median {points.confidence.median():.2f} · "
      f"min {points.confidence.min():.2f} · max {points.confidence.max():.2f}")
print(f"{'='*46}")

## 6. Check the result

**This is the step that decides whether the number is worth anything.** Do the boxes
sit on the plants? If they do not, the model does not suit your imagery - which is
common, for the reason given at the top.

In [ ]:
# @title The detections over your own image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

PATCH_PX = 1000  # @param {type:"slider", min:400, max:2000, step:100}

with rasterio.open(ORTHOPHOTO) as src:
    px = [src.index(p.x, p.y) for p in
          gpd.GeoSeries([r["centroid"] for r in records], crs=crs).to_crs(src.crs)]
    rows = [p[0] for p in px]; cols = [p[1] for p in px]
    side = min(PATCH_PX, src.width, src.height)
    c0 = int(np.clip(np.median(cols) - side//2, 0, max(0, src.width - side)))
    r0 = int(np.clip(np.median(rows) - side//2, 0, max(0, src.height - side)))
    patch = to_rgb(src.read(window=Window(c0, r0, side, side))[:3])
    gsd = abs(src.transform.a)

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(patch); ax.axis("off")
n = 0
for (ro, co), r in zip(px, records):
    if r0 <= ro < r0+side and c0 <= co < c0+side:
        half = max(6, (card["median_crown_m"] / gsd) / 2)
        ax.add_patch(Rectangle((co-c0-half, ro-r0-half), half*2, half*2,
                               fill=False, edgecolor="#f97316", linewidth=1.6))
        n += 1
ax.set_title(f"{n} detections in this {int(side*gsd)} x {int(side*gsd)} m patch", pad=14)
plt.tight_layout(); plt.show()

In [ ]:
# @title Navigable map &nbsp;·&nbsp; *satellite background, to locate the area*
import folium

c = points.geometry.union_all().centroid
m = folium.Map(location=[c.y, c.x], zoom_start=19, max_zoom=22,
               tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
               attr="Esri, Maxar")
folium.GeoJson(
    boxes,
    style_function=lambda _: {"color": "#f97316", "weight": 2, "fillOpacity": 0.15},
    tooltip=folium.GeoJsonTooltip(["class", "confidence"]),
).add_to(m)
b = boxes.total_bounds
m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
print("The satellite image is from a different date than your orthophoto -")
print("use it to locate the area, not to judge the detections.")
m

## 7. Download

In [ ]:
# @title Save GeoJSON and CSV
base = Path(ORTHOPHOTO).stem + "_" + card["species"]
boxes.to_file(base + "_boxes.geojson", driver="GeoJSON")
points.to_file(base + "_centroids.geojson", driver="GeoJSON")
tab = points.copy()
tab["longitude"], tab["latitude"] = tab.geometry.x, tab.geometry.y
tab.drop(columns="geometry").to_csv(base + ".csv", index=False)

print("All in EPSG:4326, ready for QGIS:\n")
for ext in ("_boxes.geojson", "_centroids.geojson", ".csv"):
    print("  " + base + ext)

try:
    from google.colab import files
    for ext in ("_boxes.geojson", "_centroids.geojson", ".csv"):
        files.download(base + ext)
except Exception:
    pass

---

### Running it on your own machine

This notebook is a convenience. The full tool runs **offline, without a GPU and
without sending your imagery anywhere** - which matters when the orthophoto covers a
property whose location is sensitive information:

```bash
git clone https://github.com/COURAGEOUS-LAND/agroforestry-inventory
cd agroforestry-inventory
pip install -r requirements.txt
python download_models.py
python inventory.py my_orthophoto.tif --species coffee
```

There the map uses **your orthophoto** as the background, rather than a satellite
image from another date.

Models: <https://huggingface.co/JefersonPMS/agroforestry-inventory> ·
Code: <https://github.com/COURAGEOUS-LAND/agroforestry-inventory>

Developed by **Courageous Land** · code GPL-3.0 · models CC-BY-4.0